# Week 4 · Classical vs quantum circuit model

**MIT xPRO QCFx1 · Introduction to Quantum Computing · Week 4**

Source videos: **Q1M4V1** (classical circuit model) and **Q1M4V2** (quantum circuit model).
This notebook follows `week-4-watch-guide.md`. It is a study aid, not a source of
graded answers.

Time reference: **Paramaribo (UTC−3, no DST)**. The hard course deadline is
**Mon 2026-11-02 20:30 Paramaribo**.

## What to keep in view

Both models have the same three stages:

1. **Initialize** the bits or qubits.
2. **Compute** with gates.
3. **Measure** the result.

The classical model uses irreversible gates such as AND and XOR. The quantum
model uses unitary, reversible gates, and measurement is the only irreversible
step.

## 1. Classical circuit model

The video's full-adder example has inputs $B_1, B_2, B_3$ (two data bits and a
carry-in) and outputs

$$
\text{sum} = B_1 \oplus B_2 \oplus B_3,
\qquad
\text{carry} = B_1B_2 \oplus B_1B_3 \oplus B_2B_3.
$$

The carry expression is the majority function: it is 1 when at least two inputs
are 1. The circuit writes the sum and carry into fresh output bits, which is why
the classical model includes a reset-to-zero step before the computation.

Let us enumerate the truth table. This is ordinary Python, not a quantum
simulation.

In [ ]:
from itertools import product

print("B1 B2 B3 | sum carry")
print("---------+----------")
for b1, b2, b3 in product([0, 1], repeat=3):
    total = b1 + b2 + b3
    sum_bit = total % 2
    carry = 1 if total >= 2 else 0
    print(f" {b1}  {b2}  {b3} |  {sum_bit}    {carry}")

### Why classical gates lose information

AND and XOR are not reversible. From the output `sum = 0` alone you cannot tell
whether the inputs were `000`, `011`, `101`, or `110`. The classical circuit
throws information away. Quantum gates are unitary, so they are reversible; the
information is redistributed rather than erased. Measurement is then the step
that produces a classical bit and collapses the state.

The video also describes a controller layer: a user interface sends data and an
instruction set, the controller sets bits and runs the gates, and the measured
result comes back as data. That separation between control and processing will
return in V4.

## 2. Quantum circuit model

A qubit starts in $|0\rangle$. The Hadamard gate creates an equal superposition:

$$
H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}},
\qquad
H^{\otimes n}|0\rangle^{\otimes n}
= \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle.
$$

The normalization factors are therefore

| Qubits | Amplitude on each basis state | Number of terms |
|---:|---:|---:|
| 1 | $1/\sqrt{2}$ | 2 |
| 2 | $1/2$ | 4 |
| 3 | $1/(2\sqrt{2})$ | 8 |

Measurement projects onto the computational basis. The probability of outcome
$x$ is $|\alpha_x|^2$, and one classical bit is produced per qubit.

In [ ]:
from pathlib import Path
import sys

def find_week4_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "week-4", Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "week4_helpers.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repo root, week-4/, or week-4/notebooks/."
    )

WEEK4_DIR = find_week4_dir()
if str(WEEK4_DIR) not in sys.path:
    sys.path.insert(0, str(WEEK4_DIR))

print(f"Week 4 folder: {WEEK4_DIR}")

import math
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit
import qiskit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram
import week4_helpers as w4

print(f"Qiskit version: {qiskit.__version__}")

### Exact statevector for one, two, and three Hadamards

`week4_helpers.print_statevector()` removes final measurements, builds the
statevector, and prints only the non-zero amplitudes using exact-looking forms
such as $1/\sqrt{2}$ and $1/2$.

In [ ]:
for num_qubits in (1, 2, 3):
    circuit = QuantumCircuit(num_qubits)
    circuit.h(range(num_qubits))
    print(f"\n{num_qubits} qubit(s), H on every qubit")
    w4.print_statevector(circuit)

### From amplitudes to measurement counts

The ideal probabilities for three Hadamards are all $1/8$. A finite number of
shots gives counts that fluctuate around those probabilities; the counts are
**not** the probabilities themselves. This distinction matters for the Week 2
quiz material and for hardware runs.

In [ ]:
circuit = QuantumCircuit(3)
circuit.h(range(3))
circuit.measure_all()

counts = w4.run_counts(circuit, shots=4096, seed=42)
print("Counts:", dict(sorted(counts.items())))
print("Probabilities:", {key: round(value, 4) for key, value in w4.probabilities(counts, 4096).items()})
display(plot_histogram(counts, title="Three Hadamards, 4096 shots"))

### Interference: a relative phase becomes measurable

A phase by itself is invisible in the computational basis. The Hadamard basis
turns it into a measurable difference. Compare these two one-qubit circuits:

- $H \to H$ returns $|0\rangle$.
- $H \to Z \to H$ returns $|1\rangle$, because $HZH = X$ and $X|0\rangle = |1\rangle$.

This is the same interference mechanism that Deutsch-Jozsa uses in V3.

In [ ]:
def statevector_rows(label: str, circuit: QuantumCircuit) -> None:
    print(f"\n{label}")
    w4.print_statevector(circuit)

hh = QuantumCircuit(1)
hh.h(0)
hh.h(0)

hzh = QuantumCircuit(1)
hzh.h(0)
hzh.z(0)
hzh.h(0)

statevector_rows("H then H", hh)
statevector_rows("H then Z then H", hzh)

## 3. A quantum full adder: initialize → compute → measure

The classical full adder can be made reversible by keeping the inputs and using
ancillas. We use five qubits:

| Qubit | Role |
|---:|---|
| `q[0]` | $B_1$ |
| `q[1]` | $B_2$ |
| `q[2]` | carry-in $B_3$ |
| `q[3]` | sum output |
| `q[4]` | carry output |

The sum is three CNOTs into `q[3]`. The carry is the majority function, built
from three Toffoli (CCX) gates into `q[4]`. Inputs are set with `X` gates; the
outputs are the only qubits measured. This is a teaching example, not the graded
Week 4 lab.

In [ ]:
def quantum_full_adder(b1: int, b2: int, b3: int) -> QuantumCircuit:
    circuit = QuantumCircuit(5, 2)
    if b1:
        circuit.x(0)
    if b2:
        circuit.x(1)
    if b3:
        circuit.x(2)

    # Sum = B1 XOR B2 XOR B3, written into q[3].
    circuit.cx(0, 3)
    circuit.cx(1, 3)
    circuit.cx(2, 3)

    # Carry = majority(B1, B2, B3), written into q[4].
    circuit.ccx(0, 1, 4)
    circuit.ccx(0, 2, 4)
    circuit.ccx(1, 2, 4)

    # c[0] is the sum (q[3]); c[1] is the carry (q[4]).
    circuit.measure([3, 4], [0, 1])
    return circuit


print("inputs (B1,B2,B3) -> counts; bitstring is carry-sum")
for inputs in [(0, 0, 0), (0, 1, 1), (1, 1, 0), (1, 1, 1)]:
    circuit = quantum_full_adder(*inputs)
    counts = w4.run_counts(circuit, shots=1024, seed=42)
    print(f"{inputs} -> {counts}")

For `(1, 1, 0)`, the sum is 0 and the carry is 1, so the measured bitstring is
`10` (carry on the left, sum on the right). For `(1, 1, 1)`, both are 1 and the
bitstring is `11`.

## 4. Predict before you run

1. What are the ideal probabilities for four Hadamards measured in the
   computational basis?
2. What does $HXH$ do to $|0\rangle$? Check your answer with a statevector.
3. In the full adder, why does the circuit keep the input qubits instead of
   overwriting them?
4. If a circuit has a relative phase on one branch, why can a final Hadamard
   make that phase visible?

Write your prediction first, then run the corresponding cell. The value of the
exercise is the comparison, not the printed number.